In [38]:
# imports
import kagglehub
from kagglehub import KaggleDatasetAdapter
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset
import os
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler

Matplotlib is building the font cache; this may take a moment.


In [39]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(kagglehub.dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [40]:
print(oreo_path)
print(not_oreo_path)

../data/oreo
../data/not_oreo


In [41]:
# preprocessing

# Resize, Augment, and Normalize RGB
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2), # random brightness
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=local_data_path, transform=transform)
print(type(full_dataset))
print(full_dataset)
print(f"classes found: {full_dataset.classes}")
print(f"mapping: {full_dataset.class_to_idx}")

train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)
print(type(train_loader))
print(train_loader)

<class 'torchvision.datasets.folder.ImageFolder'>
Dataset ImageFolder
    Number of datapoints: 10176
    Root location: ../data
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.5)
               RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
               ColorJitter(brightness=(0.8, 1.2), contrast=None, saturation=None, hue=None)
               ToTensor()
           )
classes found: ['not_oreo', 'oreo']
mapping: {'not_oreo': 0, 'oreo': 1}
<class 'torch.utils.data.dataloader.DataLoader'>


In [42]:
# Split

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

# 80% Train
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

# 10% validation
# 10% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=targets[temp_idx],
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

print(f"Train Size: {len(train_dataset)}")
print(f"Val Size: {len(val_dataset)}")
print(f"Test Size: {len(test_dataset)}")

Train Size: 8140
Val Size: 1018
Test Size: 1018


In [43]:
# Weighted Random Sampler

train_targets = targets[train_idx]
class_sample_count = np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_targets])
samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
oreo_count = (labels == full_dataset.class_to_idx['oreo']).sum().item()
not_oreo_count = (labels == full_dataset.class_to_idx['not_oreo']).sum().item()

print(f"Batch size: {len(labels)}")
print(f"Oreo images: {oreo_count}")
print(f"Not Oreo images: {not_oreo_count}")

Batch size: 32
Oreo images: 12
Not Oreo images: 20


In [44]:
# logistic Regression

In [45]:
# logistic validation

In [46]:
# K-NN
class KNN:
    def __init__(self, k):
        self.k = k

    # Stores training data and labels for prediction
    def store(self, x, y):
        self.x_train = x
        self.y_train = y

    # Use euclidean distance to find nearest K neighbors and predict if Oreo or not
    def predict(self, x):
        test_squared = np.sum(x**2, axis=1, keepdims=True)
        train_squared = np.sum(self.x_train**2, axis=1)

        distances = np.sqrt(np.maximum(test_squared + train_squared - 2 * np.dot(x, self.x_train.T), 0))

        # Finds majority label among K nearest neighbors
        predictions = []
        for i in range(distances.shape[0]):
            knn_indices = np.argsort(distances[i])[:self.k]
            knn_labels = self.y_train[knn_indices]
            predicted_label = np.bincount(knn_labels).argmax()
            predictions.append(predicted_label)

        return np.array(predictions)

knn = KNN(5)
knn.store(x_train, y_train)

predictions = knn.predict(x_test)

print(f"Predictions: {predictions}")
print(f"Actual: {y_test}")

NameError: name 'x_train' is not defined

In [ ]:
# K-NN validation


In [ ]:
# CNN

In [ ]:
# CNN validation

In [ ]:
# evaluation

# classification errors
def get_classification_errors(test_answers, predictions):
    TP, TN, FP, FN = 0, 0, 0, 0

    for i in range(test_answers):
        if predictions[i] == test_answers[i]:
            if test_answers[i] == 1:
                TP += 1
            else:
                TN += 1
        else:
            if test_answers[i] == 0 and predictions[i] == 1:
                FP += 1
            else:
                FN += 1

    return TP, TN, FP, FN

# Accuracy
def get_accuracy(TP, TN, FP, FN):
    return (TP + TN) / (TP + TN + FP + FN)

# Precision
def get_precision(TP , FP):
    return TP / (TP + FP)

# Recall
def get_recall(TP, FN):
    return TP / (TP + FN)

# F1 value
def get_f1_score(precision , recall):
    return 2 / (1 / precision + 1 / recall)

# Specificity
def get_specificity(TN, FP):
    return TN / (TN + FP)

# Matthews Correlation Coefficient
def get_MCC(TP, TN, FP, FN):
    numerator = TP * TN - FP * FN
    denominator = np.sqrt((TP + FP)*(TP + FN)*(TN + FP)*(TN + FN))
    return numerator / denominator

# Validation Evaluation
def validation_eval(model):
    predictions = model.predict(test_dataset)

    TP, TN, FP, FN = get_classification_errors(predictions, test_dataset)
    accuracy = get_accuracy(TP, TN, FP, FN)
    precision = get_precision(TP, FP)
    recall = get_recall(TP, FN)
    f1_score = get_f1_score(precision, recall)
    specificity = get_specificity(TN, FP)
    mcc = get_MCC(TP, TN, FP, FN)

    # Figure out what we want to return to test validations
    return f1_score

# Final Evaluation
def final_eval(model):
    predictions = model.predict(test_dataset)

    TP, TN, FP, FN = get_classification_errors(predictions, test_dataset)
    accuracy = get_accuracy(TP, TN, FP, FN)
    precision = get_precision(TP, FP)
    recall = get_recall(TP, FN)
    f1_score = get_f1_score(precision, recall)
    specificity = get_specificity(TN, FP)
    mcc = get_MCC(TP, TN, FP, FN)

    print(f"""
        True Positives: {TP}
        True Negatives: {TN}
        False Positives: {FP}
        False Negatives: {FN}
        Model Accuracy: {accuracy:.3f}
        Model Precision: {precision:.3f}
        Model Recall: {recall:.3f}
        Model F1 Score: {f1_score:.3f}
        Model Specificity: {specificity:.3f}
        Model MCC: {mcc:.3f}
        """)